In [ ]:
Задача 2.1
(5 поени) Зошто за алгоритамот ID3 за градење на дрво се вели дека е алчен алгоритам?

Одговор: Алгоритамот ID3 се карактеризира како алчен (greedy) бидејќи при секое разгранување на дрвото донесува одлука која е моментално (локално) најоптимална, односно го избира оној атрибут кој носи најголема информациска добивка (Information Gain). Притоа, тој не ја зема предвид крајната структура на дрвото и нема механизам за враќање назад (backtracking). Тоа значи дека доколку некој претходен избор се покаже како неповолен за глобалната точност подоцна, алгоритамот не може да го поправи, туку продолжува напред со веќе направениот избор.

In [ ]:
Задача 2.2
(5 поени) Подели го податочното множество на два дела. Поголемиот дел нека е 75% и ќе служи за тренирање, а помалиот дел од 25% ќе служи за проверка.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Вчитување на матрицата со податоци
dataset = pd.read_csv('rice (1).csv')

# 2. Раздвојување на влезните карактеристики од целната класа
# Колоната 'Class' е лабелата која ја предвидуваме
X_features = dataset.drop(columns=['Class'])
y_labels = dataset['Class']

# 3. Креирање на тренинг и тест множества (75% наспроти 25%)
# Поставуваме фиксен random_state за репродуктивност
train_X, test_X, train_y, test_y = train_test_split(X_features, y_labels, test_size=0.25, random_state=101)

print(f"Големина на множество за тренинг: {train_X.shape[0]} примероци")
print(f"Големина на множество за тест: {test_X.shape[0]} примероци")

In [ ]:
Задача 2.3
(5 поени) Исцртај ги на график зрната ориз така што на x оската ќе биде површината (Area), на y оската ќе биде периметарот (Perimeter), а видот на зрната ќе биде претставен со боја или симбол.

In [ ]:
import plotly.express as px

# Креирање на scatter график според упатствата од документацијата на Plotly
scatterplot = px.scatter(
    dataset, 
    x="Area", 
    y="Perimeter", 
    color="Class",
    title="Просторен приказ на зрната ориз (Area vs Perimeter)",
    labels={"Area": "Површина на зрното (Area)", "Perimeter": "Периметар на зрното (Perimeter)"}
)

scatterplot.show()

In [ ]:
Задача 2.4
(5 поени) Користејќи ја библиотеката sklearn, вчитај класификатор - дрво за одлучување, претставен преку класата DecisionTreeClassifier, а потоа вметни ги податоците за тренирање од вториот чекор за да ја истренираш мрежата.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Иницијализација на моделот
tree_estimator = DecisionTreeClassifier(random_state=101)

# Тренирање со фитање врз тренинг податоците
tree_estimator.fit(train_X, train_y)

In [ ]:
Задача 2.5
(5 поени) Пресметај ја прецизноста на алгоритамот за податоците кои тој ги нема видено (на кои нема тренирано). Потоа пресметај ја прецизноста на алгоритамот за податоците кои ги има видено (на кои има тренирано). Споредете ги резултатите. Ви изгледаат ли во ред?

In [ ]:
from sklearn.metrics import accuracy_score

# Евалуација и предвидување за двете множества
predictions_train = tree_estimator.predict(train_X)
predictions_test = tree_estimator.predict(test_X)

# Пресметка на точност
acc_train = accuracy_score(y_true=train_y, y_pred=predictions_train)
acc_test = accuracy_score(y_true=test_y, y_pred=predictions_test)

print(f"Точност на тренинг податоци (видени): {acc_train:.4f}")
print(f"Точност на тест податоци (невидени): {acc_test:.4f}")

In [ ]:
Компарација и заклучок:
Резултатите се целосно очекувани и логични. Точноста на тренинг множеството е максимална ($1.0$ или $100\%$), бидејќи кај чистите дрва на одлучување алгоритамот се разгранува сè додека совршено не ги класифицира сите достапни тренинг примери. Точноста на тест множеството е нешто пониска (обично помеѓу $87\%$ и $93\%$), што е нормално во машинското учење бидејќи се работи за нови податоци. Бидејќи разликата не е абнормално голема, моделот има добра генерализација и нема изразен штетен overfitting.

In [ ]:
Задача 2.6
(5 поени) (код) Кои зрна ориз алгоритамот успева да ги распознава подобро, големите (Area над просекот) или малите (Area под просекот)?

In [ ]:
# 1. Наоѓање на средната вредност за Area во тест множеството
average_area_value = test_X['Area'].mean()

# 2. Филтрирање на индексите на големи и мали зрна во тест множеството
big_grains_idx = test_X[test_X['Area'] > average_area_value].index
small_grains_idx = test_X[test_X['Area'] <= average_area_value].index

# 3. Изолирање на вистинските класи и предвидувањата за двете подмножества
# За правилно мапирање, ги ставаме предвидувањата во Pandas серија со соодветни индекси
predictions_series = pd.Series(predictions_test, index=test_X.index)

accuracy_big = accuracy_score(test_y.loc[big_grains_idx], predictions_series.loc[big_grains_idx])
accuracy_small = accuracy_score(test_y.loc[small_grains_idx], predictions_series.loc[small_grains_idx])

print(f"Прецизност кај големи зрна (над просекот): {accuracy_big:.4f}")
print(f"Прецизност кај мали зрна (под просекот): {accuracy_small:.4f}")

if accuracy_big > accuracy_small:
    print("Заклучок: Моделот е поефикасен во препознавање на ГОЛЕМИТЕ зрна ориз.")
else:
    print("Заклучок: Моделот е поефикасен во препознавање на МАЛИТЕ зрна ориз.")

In [ ]:
Задача 2.7
(5 поени) Од кој вид се зрната ориз од податочното множество од датотеката rice_test.csv?

In [ ]:
# Вчитување на надворешната датотека за евалуација
external_test_df = pd.read_csv('rice_test.csv')

# Класификација со помош на веќе тренираното дрво
final_predictions = tree_estimator.predict(external_test_df)

print("Резултати од класификацијата за rice_test.csv:")
print(final_predictions)

In [ ]:
Задача 2.8
(5 поени) Дали постапуваме правилно ако за нашите податоци имплементираме многу различни алгоритми, на пример над 20, и го избереме најдобриот? Дали сигурно сме го избрале најдобриот или има нешто труло во оваа постапка? Објасни.

Одговор:
Оваа постапка не е правилна и крие сериозна статистичка стапица позната како Data Snooping Bias или Проблем на повеќекратни споредби (Multiple Comparisons Problem).

Кога без претходна ригорозна методологија ќе тестираме огромен број на алгоритми (над 20) на едно исто тест множество, веројатноста дека еден од нив ќе покаже одличен резултат чисто поради среќа или случајно поклопување со шумот на тие конкретни податоци е енормно висока. На тој начин ние всушност правиме "overfitting" на самиот процес на селекција. Моделот кој изгледа најдобро на тоа множество, многу веројатно ќе потфрли во реалноста.

За да биде постапката валидна, евалуацијата мора да се изведе со помош на вкрстена валидација (K-fold Cross-Validation) на тренинг множеството, а финалниот победнички модел да се тестира на посебно, трето т.н. Hold-out тест множество кое воопшто не учествувало во изборот.